# Deployment checks and handoff

## Purpose and readiness levels

Confirm that the supplied saved model is the real-data artifact expected by the app. The code checks the training status and model checksum and prints the local launch command. These are artifact checks, not a browser interaction or a public deployment test.

The recorded verification compared 1,000 model predictions with saved evaluation outputs and checked numerical imputation across all training rows. A separate real-data batch test selected exactly 100 of 1,000 flights at 10% capacity. Eighteen software tests passed in the completed run.

Code-cell outputs are retained from the completed real-data run; this documentation update does not rerun or alter their computation.


In [1]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/schema.py').exists())
sys.path.insert(0, str(ROOT))
def run_stage(name):
    completed = subprocess.run([sys.executable, str(ROOT / 'scripts' / name)], cwd=ROOT, capture_output=True, text=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
print('Project directory:', ROOT.name)


Project directory: flight_disruption_project


## Deployment sequence

1. Confirm root paths: `requirements.txt`, `app/streamlit_app.py`, `src/`, the three model files and required report tables.
2. Install dependencies with Python 3.12 and launch Streamlit using the supplied artifact; full training data is not needed for app inference.
3. Exercise valid and invalid individual inputs, a batch CSV, capacity selection and download.
4. Deploy from `Celestica19/cds4003-flight-disruption`, branch `main`, main file `app/streamlit_app.py`.
5. Verify the deployed app while signed out, then record its URL and deployed versions.

Use the [GitHub update guide](../docs/github_update_guide.md) to correct nested folders and the [run guide](../docs/run_and_deploy.md) for the acceptance-check table.


In [2]:
meta = json.loads((ROOT / 'models/metadata.json').read_text())
assert meta['status'] == 'TRAINED_REAL_DATA', 'Fixture models may not be deployed'
from src.common import sha256_file
assert sha256_file(ROOT / 'models/model.joblib') == meta['model_sha256']
print('Model:', meta['winner'])
print('Local launch command: python -m streamlit run app/streamlit_app.py')
print('Public URL and repository URL must be added after deployment.')

Model: hist_gradient_boosting
Local launch command: python -m streamlit run app/streamlit_app.py
Public URL and repository URL must be added after deployment.


## Remaining evidence

Streamlit installation and actual UI execution were unavailable in the analysis environment. Public deployment is still pending. The repository exists, but the revised root upload needs verification. A successful checksum check alone does not close either item.

Capture genuine deployment evidence only after running the app: a valid prediction, invalid-input feedback, batch review/download, displayed metrics and signed-out access. Keep screenshots free of account secrets and private browser information. Record model limitations and the live URL in the README after verification.
